# Supreme Court Judgment Preprocessing & Automatic Annotation
**Raw TXT → cleaned judgment → outcome extraction → pre-analysis model input → annotations**

This is a conservative pilot pipeline for Google Colab and Google Drive.

### Outputs
- `cleaned_text/`: normalized full judgment text
- `model_input/`: facts/proceedings/party submissions before the court's own analysis, when a boundary is detected
- `master_annotations.csv`: one row per judgment
- `outcome_evidence.jsonl`: evidence and detected outcome
- `review_queue.csv`: uncertain, incomplete, or boundary-ambiguous documents
- `processing_errors.csv`: files that failed

**Important:** Automatic labels are silver labels, not ground truth. Run a small pilot, inspect outputs, and only then scale up. Keep raw files unchanged.

## 1. Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Configure folders and processing options

In [3]:
from pathlib import Path

# CHANGE THIS to the folder containing your original .txt judgment files.
RAW_DIR = Path('/content/drive/MyDrive/CJPE/raw_text')

# Output folder. Keep it separate from RAW_DIR.
OUTPUT_DIR = Path('/content/drive/MyDrive/CJPE/processed_v5')

# Optional: process only the first N files for a pilot. Set to None for all files.
MAX_FILES = 100

# If your filenames contain a stable Indian Kanoon document ID, keep this True.
# Example: 198100584.txt -> doc_id 198100584
USE_FILENAME_AS_DOC_ID = True


CLEAN_DIR = OUTPUT_DIR / 'cleaned_text'
MODEL_DIR = OUTPUT_DIR / 'model_input'
for folder in [OUTPUT_DIR, CLEAN_DIR, MODEL_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print('Raw folder exists:', RAW_DIR.exists())
print('Output folder:', OUTPUT_DIR)

Raw folder exists: True
Output folder: /content/drive/MyDrive/CJPE/processed_v5


## 3. Imports and conservative text normalization

The source text often has page numbers, signature overlays, soft hyphens, and line wrapping. The cleaner below:
- removes only isolated page-number and signature-metadata lines;
- removes the signature watermark phrase even when it is embedded in a judgment line, without deleting the rest of that line;
- joins wrapped lines while preserving numbered paragraph boundaries and headings;
- does not rewrite legal wording or attempt OCR repair.

Inspect the cleaned pilot output before processing the full corpus.

In [4]:
import re, csv, json, hashlib, traceback
import pandas as pd
from collections import Counter
from datetime import datetime

PARA_MARKER = re.compile(r'^\s*(?:\[\d+(?:\.\d+)*\]|\(\d+\))\s*')
HEADING_LINE = re.compile(
    r'^\s*(?:JUDGMENT|JUDGEMENTS?|ORDER|OPERATIVE ORDER|FINAL ORDER|'
    r'CONCLUSION|RESULT|APPEARANCES|REPORTABLE|NON-REPORTABLE)\s*$',
    re.IGNORECASE
)
SIGNATURE_LINE = re.compile(
    r'^\s*(?:Digitally\s+signed\s+by\b.*|'
    r'Date:\s*\d{4}[.-]\d{2}[.-]\d{2}.*|'
    r'Reason:\s*.*|'
    r'\d{1,2}:\d{2}:\d{2}\s+IST)\s*$',
    re.IGNORECASE
)

def normalize_text(text):
    # Normalize Unicode whitespace and line endings; remove invisible artifacts.
    text = text.replace('\r\n', '\n').replace('\r', '\n')
    text = text.replace('\u00a0', ' ')
    text = text.replace('\u00ad', '')  # discretionary soft hyphen
    text = re.sub(r'[\u200b\ufeff]', '', text)

    # Signature watermark can be inserted in the middle of a legitimate line.
    text = re.sub(r'\bSignature\s+Not\s+Verified\b', '', text, flags=re.IGNORECASE)

    output_paragraphs = []
    current_lines = []

    def flush():
        nonlocal current_lines
        if current_lines:
            paragraph = ' '.join(x.strip() for x in current_lines if x.strip())
            paragraph = re.sub(r'[ \t]+', ' ', paragraph).strip()
            if paragraph:
                output_paragraphs.append(paragraph)
        current_lines = []

    for raw_line in text.splitlines():
        line = raw_line.strip()
        if not line:
            continue

        # Isolated page numbers from Indian Kanoon/PDF extraction.
        if re.fullmatch(r'\d{1,4}', line):
            continue

        # Isolated digital-signature metadata and signer names.
        if SIGNATURE_LINE.match(line):
            continue
        if re.fullmatch(r'[A-Z][A-Za-z .-]{1,40}', line) and line.lower() in {
            'natarajan', 'digitally signed'
        }:
            continue

        # Start a new paragraph at explicit paragraph numbers/headings.
        if PARA_MARKER.match(line) or HEADING_LINE.match(line):
            flush()
            current_lines.append(line)
        else:
            current_lines.append(line)

    flush()
    return '\n\n'.join(output_paragraphs).strip()


def get_doc_id(path):
    stem = path.stem
    if USE_FILENAME_AS_DOC_ID:
        return stem
    return hashlib.sha1(str(path).encode()).hexdigest()[:12]


def extract_year(text, path):
    m = re.search(
        r'(?:on\s+)?(\d{1,2}\s+[A-Za-z]+\s*,?\s+20\d{2})',
        text[:4000],
        flags=re.IGNORECASE
    )
    if m:
        y = re.search(r'20\d{2}', m.group(1))
        if y:
            return int(y.group())
    y = re.search(r'20\d{2}', str(path))
    return int(y.group()) if y else None


In [5]:
# Cleaner sanity checks: ordinary words must survive; page numbers and
# signature overlays must not survive as standalone artifacts.
_sample = (
    "Supreme Court of India\n"
    "JUDGMENT\n"
    "[1.0] Appeal is dismissed after hearing the parties.\n"
    "3\n"
    "Signature Not Verified\n"
    "Natarajan\n"
    "17:01:55 IST\n"
)
_sample_clean = normalize_text(_sample)
for word in ["Supreme", "Court", "JUDGMENT", "Appeal", "dismissed"]:
    assert word in _sample_clean, f"Cleaner deleted expected word: {word}"
assert not re.search(r'(?m)^\s*3\s*$', _sample_clean)
assert "Signature Not Verified" not in _sample_clean
assert "17:01:55 IST" not in _sample_clean
print("Cleaner sanity checks passed.")
print(_sample_clean)


Cleaner sanity checks passed.
Supreme Court of India

JUDGMENT

[1.0] Appeal is dismissed after hearing the parties.


## 4. Detect the end of the judgment and the start of judicial analysis

The final-order detector looks for an outcome statement in the last few paragraphs, rather than any mention of an outcome anywhere in the final quarter. If the order is missing or cannot be identified, the document is sent for review.

For prediction input, the notebook separately looks for the beginning of the court's own analysis (for example, `Heard learned counsel`). It retains only the preceding case background and submissions. If that boundary cannot be identified confidently, the model input is left blank instead of silently including the court's reasoning.

In [6]:
def split_paragraphs(text):
    return [p.strip() for p in re.split(r'\n\s*\n+', text) if p.strip()]


ANALYSIS_START_PATTERNS = [
    r'^\s*(?:\[\d+(?:\.\d+)*\]\s*)?Heard\s+(?:learned|Ld\.?|the)\s+',
    r'^\s*(?:\[\d+(?:\.\d+)*\]\s*)?We\s+have\s+heard\b',
    r'^\s*(?:\[\d+(?:\.\d+)*\]\s*)?Having\s+heard\b',
    r'^\s*(?:\[\d+(?:\.\d+)*\]\s*)?We\s+have\s+considered\b',
    r'^\s*(?:\[\d+(?:\.\d+)*\]\s*)?The\s+question\s+(?:which|that)\s+(?:arises|falls)',
    r'^\s*(?:\[\d+(?:\.\d+)*\]\s*)?The\s+short\s+question\b',
]
ANALYSIS_START_RE = re.compile(
    '|'.join(f'(?:{p})' for p in ANALYSIS_START_PATTERNS),
    re.IGNORECASE | re.MULTILINE
)

FINAL_HEADINGS = re.compile(
    r'(?im)^\s*(?:OPERATIVE\s+(?:ORDER|PART)|FINAL\s+ORDER|'
    r'ORDER|RESULT|CONCLUSION|DISPOSAL|DISPOSITION)\s*[:.]?\s*$'
)

# These patterns are deliberately stricter than a generic mention of "allowed"
# or "dismissed"; references to earlier cases should not determine this case's label.
FINAL_OUTCOME_RE = re.compile(
    r'\b(?:the\s+)?(?:present\s+)?(?:civil\s+)?(?:appeal|appeals|'
    r'petition|petitions|application|applications)\s+'
    r'(?:(?:is|are|stands|stand)\s+)?(?:hereby\s+|accordingly\s+)?'
    r'(?:partly\s+allowed|allowed|dismissed|withdrawn)\b'
    r'|\b(?:appeal|petition|application)\s+stands\s+disposed\b'
    r'|\b(?:appeal|petition|application)\s+is\s+disposed\s+of\b',
    re.IGNORECASE
)

def locate_analysis_start(text):
    m = ANALYSIS_START_RE.search(text)
    if not m:
        return None, 'analysis_start_not_found'
    return m.start(), 'analysis_start_phrase'


def locate_final_region(text):
    paras = split_paragraphs(text)
    if not paras:
        return None, [], 'empty'

    # Explicit final-order heading near the end.
    tail_start = int(len(text) * 0.65)
    heading = FINAL_HEADINGS.search(text[tail_start:])
    if heading:
        return tail_start + heading.start(), paras, 'final_heading'

    # Search only the last 8 paragraphs for a clear disposition statement.
    # The last matching paragraph is preferred to avoid earlier case citations.
    offsets = []
    cursor = 0
    for para in paras:
        pos = text.find(para, cursor)
        offsets.append(pos)
        cursor = pos + len(para)

    for para, pos in reversed(list(zip(paras[-8:], offsets[-8:]))):
        if FINAL_OUTCOME_RE.search(para):
            return pos, paras, 'last_paragraph_disposition'

    return None, paras, 'final_order_not_found'


def extract_tail(text, max_chars=12000):
    return text[-max_chars:] if len(text) > max_chars else text


## 5. Extract outcome evidence from the final region

The outcome is extracted only when a final-order region is found. If the judgment is incomplete, the final order is absent, or the wording is ambiguous, the outcome remains `unclear` and is placed in the review queue.

In [7]:
OUTCOME_PATTERNS = {
    'partly_allowed': [
        r'\b(?:appeal|appeals|petition|petitions|application|applications)\s+'
        r'(?:(?:is|are|stands|stand)\s+)?(?:hereby\s+|accordingly\s+)?partly\s+allowed\b',
        r'\bpartly\s+allow(?:ed|s)\b'
    ],
    'allowed': [
        r'\b(?:appeal|appeals|petition|petitions|application|applications)\s+'
        r'(?:(?:is|are|stands|stand)\s+)?(?:hereby\s+|accordingly\s+)?allowed\b'
    ],
    'dismissed': [
        r'\b(?:appeal|appeals|petition|petitions|application|applications)\s+'
        r'(?:(?:is|are|stands|stand)\s+)?(?:hereby\s+|accordingly\s+)?dismissed\b'
    ],
    'remanded': [
        r'\b(?:matter|case|appeal|appeals|proceedings)\b.{0,100}\bremand(?:ed)?\b',
        r'\bremand(?:ed)?\s+to\s+(?:the\s+)?(?:High Court|Tribunal|court|authority)\b'
    ],
    'withdrawn': [
        r'\b(?:appeal|petition|application)\b.{0,80}\bwithdrawn\b',
        r'\bwithdrawn\s+(?:with|and)\s+liberty\b'
    ],
    'disposed_other': [
        r'\b(?:disposed of|stands disposed|are disposed|is disposed)\b'
    ]
}

def sentence_candidates(text):
    # The cleaner has already joined wrapped lines into paragraph-sized blocks.
    parts = re.split(r'(?<=[.!?])\s+(?=[A-Z“"(])', text)
    return [s.strip() for s in parts if s.strip()]


def find_outcome_evidence(final_text):
    hits = []
    for sentence in sentence_candidates(final_text):
        sentence_hits = []
        for label, patterns in OUTCOME_PATTERNS.items():
            if any(re.search(p, sentence, flags=re.IGNORECASE | re.DOTALL) for p in patterns):
                sentence_hits.append(label)
        # Reject explicit negation of the detected disposition.
        if re.search(
            r'\b(?:not|never|cannot|could not|would not)\s+(?:be\s+)?'
            r'(?:allowed|dismissed|remanded|withdrawn)\b',
            sentence,
            flags=re.IGNORECASE
        ):
            continue
        for label in sentence_hits:
            hits.append({'label': label, 'evidence_text': sentence[:1200]})
    return hits


def classify_outcomes(hits, final_region_found):
    labels = {h['label'] for h in hits}
    if not final_region_found or not labels:
        return 'unclear', 'low'

    substantive = labels - {'disposed_other'}
    if 'partly_allowed' in substantive and len(substantive) == 1:
        return 'partly_allowed', 'medium'
    if 'allowed' in substantive and 'dismissed' in substantive:
        return 'mixed', 'medium'
    if 'allowed' in substantive and 'remanded' in substantive:
        return 'mixed', 'medium'
    if 'partly_allowed' in substantive and len(substantive) > 1:
        return 'mixed', 'medium'
    if substantive == {'allowed'}:
        return 'allowed', 'medium'
    if substantive == {'dismissed'}:
        return 'dismissed', 'medium'
    if substantive == {'remanded'}:
        return 'remanded', 'medium'
    if substantive == {'withdrawn'}:
        return 'withdrawn', 'medium'
    if not substantive:
        return 'disposed_other', 'low'
    return 'mixed', 'low'


def ildc_labels(detailed_outcome, confidence):
    # Strict convention: mixed/partial/remand/unclear are not binary single labels.
    if confidence == 'low':
        return None, None
    if detailed_outcome == 'allowed':
        return 1, 1
    if detailed_outcome == 'dismissed':
        return 0, 0
    if detailed_outcome in ('partly_allowed', 'mixed'):
        return None, 1
    return None, None


## 6. Create model input without the court's own analysis

For an outcome-prediction task, the model input should contain the background, procedural history, and party submissions—not the court's reasoning or final order. The notebook therefore cuts at the first recognized analysis-start phrase.

This is still heuristic. If the boundary cannot be found, the model input is left blank and the document is sent to review. Manually audit the pilot; do not treat automatic boundary detection as a guarantee.

In [8]:
def remove_court_analysis(cleaned_text, analysis_start):
    if analysis_start is None:
        return '', 'analysis_removal_not_confirmed'
    model_text = cleaned_text[:analysis_start].strip()
    if len(model_text) < 500:
        return '', 'analysis_removal_too_short'
    return model_text, 'truncated_before_court_analysis'


## 7. Process all raw TXT files and generate records

In [9]:
if not RAW_DIR.exists():
    raise FileNotFoundError(
        f'RAW_DIR does not exist: {RAW_DIR}. Update the path in Section 2.'
    )

txt_files = sorted(RAW_DIR.rglob('*.txt'))
if MAX_FILES is not None:
    txt_files = txt_files[:MAX_FILES]

print('TXT files found for processing:', len(txt_files))

records = []
evidence_records = []
errors = []

for idx, path in enumerate(txt_files, start=1):
    try:
        raw = path.read_text(encoding='utf-8', errors='replace')
        cleaned = normalize_text(raw)

        final_start, paragraphs, region_method = locate_final_region(cleaned)
        final_region = cleaned[final_start:] if final_start is not None else ''
        hits = find_outcome_evidence(final_region) if final_start is not None else []

        detailed, confidence = classify_outcomes(
            hits,
            final_region_found=(final_start is not None)
        )
        single_label, multi_label = ildc_labels(detailed, confidence)

        analysis_start, analysis_method = locate_analysis_start(cleaned)
        model_text, removal_status = remove_court_analysis(cleaned, analysis_start)

        doc_id = get_doc_id(path)
        year = extract_year(cleaned, path)

        clean_path = CLEAN_DIR / f'{doc_id}.txt'
        clean_path.write_text(cleaned, encoding='utf-8')

        model_path = ''
        if model_text:
            model_path_obj = MODEL_DIR / f'{doc_id}.txt'
            model_path_obj.write_text(model_text, encoding='utf-8')
            model_path = str(model_path_obj)

        evidence_text = ' || '.join(
            dict.fromkeys(h['evidence_text'] for h in hits)
        )

        annotation_status = (
            'auto_labeled_pending_review'
            if confidence == 'medium' and removal_status == 'truncated_before_court_analysis'
            else 'needs_review'
        )

        record = {
            'doc_id': doc_id,
            'year': year,
            'detailed_outcome': detailed.upper(),
            'ildc_single_label': single_label,
            'ildc_multi_label': multi_label,
            'confidence': confidence,
            'matches_found': len(hits),
            'outcome_events': json.dumps(hits, ensure_ascii=False),
            'evidence_text': evidence_text,
            'source_file': str(path),
            'cleaned_file': str(clean_path),
            'model_input_file': model_path,
            'final_region_method': region_method,
            'analysis_start_method': analysis_method,
            'removal_status': removal_status,
            'annotation_status': annotation_status
        }
        records.append(record)

        evidence_records.append({
            'doc_id': doc_id,
            'source_file': str(path),
            'final_region_method': region_method,
            'final_region_found': final_start is not None,
            'final_region_text': final_region[-10000:],
            'outcome_events': hits,
            'detailed_outcome': detailed,
            'confidence': confidence,
            'analysis_start_method': analysis_method,
            'removal_status': removal_status
        })

    except Exception as e:
        errors.append({
            'source_file': str(path),
            'error': repr(e),
            'traceback': traceback.format_exc()
        })

    if idx % 100 == 0:
        print(f'Processed {idx}/{len(txt_files)}')

print('Processed:', len(records))
print('Errors:', len(errors))


TXT files found for processing: 100
Processed 100/100
Processed: 100
Errors: 0


## 8. Save master CSV, evidence JSONL, review queue, and errors

In [10]:
master_df = pd.DataFrame(records)

columns = [
    'doc_id', 'year',
    'ildc_single_label', 'ildc_multi_label',
    'detailed_outcome', 'confidence',
    'matches_found', 'outcome_events', 'evidence_text',
    'source_file', 'cleaned_file', 'model_input_file',
    'final_region_method', 'analysis_start_method',
    'removal_status', 'annotation_status'
]
master_df = master_df.reindex(columns=columns)

master_csv = OUTPUT_DIR / 'master_annotations.csv'
master_df.to_csv(master_csv, index=False, encoding='utf-8-sig')

with open(OUTPUT_DIR / 'outcome_evidence.jsonl', 'w', encoding='utf-8') as f:
    for item in evidence_records:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

review_df = master_df[
    (master_df['annotation_status'] == 'needs_review') |
    (master_df['confidence'] == 'low') |
    (master_df['removal_status'] != 'truncated_before_court_analysis') |
    (master_df['final_region_method'] == 'final_order_not_found')
].copy()
review_df.to_csv(OUTPUT_DIR / 'review_queue.csv', index=False, encoding='utf-8-sig')

errors_df = pd.DataFrame(errors)
errors_df.to_csv(OUTPUT_DIR / 'processing_errors.csv', index=False, encoding='utf-8-sig')

print('Saved:', master_csv)
print('Review queue rows:', len(review_df))
print('Errors:', len(errors_df))


Saved: /content/drive/MyDrive/CJPE/processed_v5/master_annotations.csv
Review queue rows: 100
Errors: 0


## 9. Quality checks and label distribution

In [11]:
print('Rows:', len(master_df))
print('Unique doc IDs:', master_df['doc_id'].nunique())
print('Duplicate IDs:', master_df['doc_id'].duplicated().sum())
print('Missing year:', master_df['year'].isna().sum())
print('Missing outcome evidence:', master_df['evidence_text'].fillna('').eq('').sum())
print('Final order not found:',
      master_df['final_region_method'].eq('final_order_not_found').sum())
print('Analysis boundary not found:',
      master_df['analysis_start_method'].eq('analysis_start_not_found').sum())
print('Model input not generated:', master_df['model_input_file'].fillna('').eq('').sum())

print('\nDetailed outcome distribution:')
display(master_df['detailed_outcome'].value_counts(dropna=False).to_frame('documents'))

print('\nYear distribution:')
display(master_df['year'].value_counts(dropna=False).sort_index().to_frame('documents'))

print('\nAnnotation status:')
display(master_df['annotation_status'].value_counts(dropna=False).to_frame('documents'))


Rows: 100
Unique doc IDs: 100
Duplicate IDs: 0
Missing year: 0
Missing outcome evidence: 35
Final order not found: 35
Analysis boundary not found: 99
Model input not generated: 99

Detailed outcome distribution:


,documents
detailed_outcome,
ALLOWED,40
UNCLEAR,35
DISMISSED,13
DISPOSED_OTHER,6
MIXED,4
PARTLY_ALLOWED,2



Year distribution:


,documents
year,
2022,100



Annotation status:


,documents
annotation_status,
needs_review,100


## 10. Inspect individual judgments

Set a document ID to inspect its cleaned text, extracted evidence, and model input. Always compare these outputs with the original judgment before accepting labels.

In [12]:
# Replace with a document ID from master_df.
INSPECT_DOC_ID = master_df.iloc[0]['doc_id'] if len(master_df) else None

if INSPECT_DOC_ID:
    row = master_df.loc[master_df['doc_id'] == INSPECT_DOC_ID].iloc[0]
    print('DOC ID:', INSPECT_DOC_ID)
    print('OUTCOME:', row['detailed_outcome'])
    print('CONFIDENCE:', row['confidence'])
    print('EVIDENCE:', row['evidence_text'])
    print('ANALYSIS START:', row['analysis_start_method'])
    print('REMOVAL STATUS:', row['removal_status'])

    print('\n--- CLEANED TEXT (last 5000 chars) ---')
    print(Path(row['cleaned_file']).read_text(encoding='utf-8')[-5000:])

    if row['model_input_file']:
        print('\n--- MODEL INPUT (last 3000 chars) ---')
        print(Path(row['model_input_file']).read_text(encoding='utf-8')[-3000:])
    else:
        print('No model input generated; analysis boundary needs review.')


DOC ID: 101618887
OUTCOME: UNCLEAR
CONFIDENCE: low
EVIDENCE: 
ANALYSIS START: analysis_start_phrase
REMOVAL STATUS: truncated_before_court_analysis

--- CLEANED TEXT (last 5000 chars) ---
same with respect to purchase and sale of the property in case of sale and purchase of the land belonging to a Scheduled Caste person in the State of Rajasthan and when the said land was allotted to the original land owner – Chunilal as Scheduled Caste landless person.

[9] At this stage, it is required to be noted that in the subsequent decision in the case of Ranjana Kumari (supra), a Three Judge Bench of this Court had an occasion to consider the same issue. Before this Court the appellant belonged to Valmiki Caste (Scheduled Caste of the State of Punjab), who married a person belonging to Valmiki Caste of Uttarakhand migrated to that State. It was found that in the State of Uttarakhand also under the Presidential order ‘Valmiki’ was also recognized as notified Scheduled Caste. Even the State of Ut

## 11. Export a compact CSV for downstream ML

This file keeps the labels and the path to the verdict-removed input. It does not embed all judgment text in the CSV, which avoids very large CSV files.

In [13]:
ml_df = master_df[
    (master_df['model_input_file'].fillna('') != '') &
    (master_df['removal_status'] == 'truncated_before_court_analysis') &
    (master_df['confidence'] != 'low')
][[
    'doc_id', 'year', 'ildc_single_label', 'ildc_multi_label',
    'detailed_outcome', 'confidence', 'model_input_file'
]].copy()

ml_path = OUTPUT_DIR / 'ml_ready_index.csv'
ml_df.to_csv(ml_path, index=False, encoding='utf-8-sig')

print('ML index saved:', ml_path)
print('Eligible rows (requires manual audit):', len(ml_df))


ML index saved: /content/drive/MyDrive/CJPE/processed_v5/ml_ready_index.csv
Eligible rows (requires manual audit): 0


## 12. Important methodological notes

1. **Silver labels, not ground truth:** rule-based labels must be manually validated.
2. **Model input boundary:** the model input is intended to stop before the court's own analysis, not merely before the final order. Automatic phrase detection is imperfect; audit every pilot document.
3. **Incomplete judgments:** if the final order is absent or cannot be detected, the outcome is `UNCLEAR` and the document enters the review queue.
4. **Multiple appeals:** this notebook creates document-level labels. Rigorous appeal-level work needs one row per appeal and a separate outcome.
5. **Remand and partial allowance:** neither should be silently converted into a simple rejection/acceptance label.
6. **Data splitting:** split by judgment/document before chunking. Never place chunks from the same judgment in different splits.
7. **Versioning:** keep raw, cleaned, annotation, and model-input files separate. Never overwrite original downloaded TXT files.
8. **Pilot first:** inspect a representative sample of cleaned text, model inputs, and evidence before increasing `MAX_FILES` or processing the full corpus.